# Retrieval-Augmented Generation (RAG) using OpenAI

Scraping a web url and answer questions from it.

In [1]:
import os
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY")
## Langsmith Tracking
os.environ["LANGCHAIN_API_KEY"] = os.getenv("LANGCHAIN_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = os.getenv("LANGCHAIN_PROJECT")

In [2]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")
print(llm)

client=<openai.resources.chat.completions.completions.Completions object at 0x10d9d6900> async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x10e436240> root_client=<openai.OpenAI object at 0x10d005730> root_async_client=<openai.AsyncOpenAI object at 0x10d9d4380> model_name='gpt-4o' model_kwargs={} openai_api_key=SecretStr('**********')


In [3]:
## Input and get response form LLM

result = llm.invoke("What is generative AI?")

In [4]:
print(result)

content='Generative AI refers to a subset of artificial intelligence technologies that can produce new content, whether it be text, images, music, or other media, that is similar to the data on which they were trained. These models use techniques such as deep learning and neural networks to understand and mimic the patterns found in the training data. \n\nOne of the most well-known types of generative AI is the Generative Adversarial Network (GAN), which consists of two neural networks, a generator and a discriminator, that work together to create realistic outputs. Another popular approach is the use of transformer models, like GPT (Generative Pre-trained Transformer), which excel in natural language processing tasks.\n\nGenerative AI has several applications, including but not limited to:\n\n1. **Content Creation**: Automatically generating written content, artwork, music, or even video.\n2. **Design**: Creating designs or prototypes for fashion, architecture, and other domains.\n3. 

In [5]:
### Chatprompt Template
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            "You are an expert AI Engineer. Provide me answers based on the questions",
        ),
        ("user", "{input}"),
    ]
)
prompt

ChatPromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='You are an expert AI Engineer. Provide me answers based on the questions'), additional_kwargs={}), HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['input'], input_types={}, partial_variables={}, template='{input}'), additional_kwargs={})])

In [6]:
## chaining
chain = prompt | llm

response = chain.invoke({"input": "Can you tell me about Langsmith?"})
print(response)

content='Langsmith is a tool or framework designed to aid developers in the building, testing, and evaluation of applications that utilize large language models (LLMs). It emphasizes enhancing the performance and reliability of these applications by providing features such as debugging, monitoring, and testing capabilities. Langsmith is particularly useful for those using LangChain, a popular framework for developing LLM-powered applications. It allows developers to track tokens used, visualize execution traces, manage examples and datasets, and offers an intuitive interface to handle various stages of app development and maintenance. This can significantly streamline the process of iterating on and scaling applications that rely on language models.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 125, 'prompt_tokens': 33, 'total_tokens': 158, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens'

In [7]:
type(response)

langchain_core.messages.ai.AIMessage

In [8]:
## stroutput Parser

from langchain_core.output_parsers import StrOutputParser

output_parser = StrOutputParser()
chain = prompt | llm | output_parser

response = chain.invoke({"input": "Can you tell me about Langsmith?"})
print(response)

Langsmith is a powerful suite of tools developed by LangChain that is designed to enhance the development and deployment of applications based on large language models (LLMs). It provides essential features that developers need to test, evaluate, and monitor their LLM applications effectively, focusing on quality and performance management.

Key features of Langsmith include:

1. **Testing and Evaluation**: Langsmith facilitates the creation and execution of various tests to evaluate different aspects of LLM applications. This includes assessing accuracy, response quality, and overall application performance.

2. **Monitoring**: The suite provides robust monitoring capabilities, allowing developers to track the behavior of their applications in real-time or overtime. This helps in identifying issues, understanding usage patterns, and optimizing application performance.

3. **Integration**: Langsmith is designed to integrate seamlessly with other LangChain tools and potentially other th

## RAG application

> Load Data --> Docs --> Divide our Docuemnts into chunks dcouments --> text --> vectors --> Vector Embeddings ---> Vector Store DB

In [11]:
## Data Ingestion--From the website we need to scrape the data
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

In [17]:
# Load data
loader = WebBaseLoader(
    "https://docs.smith.langchain.com/tutorials/Administrators/manage_spend"
)

# convert data into chunks of documents
docs = loader.load()
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
documents = text_splitter.split_documents(docs)

# create vector embeddings
embeddings = OpenAIEmbeddings()

# store vector embeddings into vector DB
vectorstoredb = FAISS.from_documents(documents, embeddings)

In [13]:
## Query From a vector db
query = "LangSmith has two usage limits: total traces and extended"
result = vectorstoredb.similarity_search(query)
result[0].page_content

'\uf8ffü¶úÔ∏è\uf8ffüõ†Ô∏è LangSmith\n\n\n\n\n\n\n\n\nSkip to main contentOur Building Ambient Agents with LangGraph course is now available on LangChain Academy!API ReferenceRESTPythonJS/TSSearchRegionUSEUGo to AppPage Not FoundWe could not find what you were looking for.Head back to our main docs page or use the search bar to find the page you need.CommunityLangChain ForumTwitterGitHubDocs CodeLangSmith SDKPythonJS/TSMoreHomepageBlogLangChain Python DocsLangChain JS/TS DocsCopyright ¬© 2025 LangChain, Inc.'

### AI response

In [19]:
from langchain_openai import ChatOpenAI
from langchain.chains.combine_documents import create_stuff_documents_chain
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document
from langchain.chains import create_retrieval_chain
from langchain_core.output_parsers import StrOutputParser


# initilize LLM
llm = ChatOpenAI(model="gpt-4o")

# create prompt for LLM
prompt = ChatPromptTemplate.from_template(
"""
Answer the following question based only on the provided context:
<context>
{context}
</context>
"""
)

# Output parser
output_parser = StrOutputParser()

# chaining
document_chain = create_stuff_documents_chain(llm, prompt)

# create a retriever
retriever = vectorstoredb.as_retriever()
retrieval_chain = create_retrieval_chain(retriever, document_chain)



## Get the response form the LLM
response = retrieval_chain.invoke(
    {"input": "LangSmith has two usage limits: total traces and extended"}
)
print(response["answer"])

Based on the provided context, the "Building Ambient Agents with LangGraph" course is available on LangChain Academy. There are also mentions of resources like the LangSmith SDK for Python and JS/TS, as well as community links to the LangChain Forum, Twitter, and GitHub.
